##### Copyright 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow 2 quickstart for beginners

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/quickstart/beginner"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This short introduction uses [Keras](https://www.tensorflow.org/guide/keras/overview) to:

1. Load a prebuilt dataset.
1. Build a neural network machine learning model that classifies images.
2. Train this neural network.
3. Evaluate the accuracy of the model.

This tutorial is a [Google Colaboratory](https://colab.research.google.com/notebooks/welcome.ipynb) notebook. Python programs are run directly in the browser—a great way to learn and use TensorFlow. To follow this tutorial, run the notebook in Google Colab by clicking the button at the top of this page.

1. In Colab, connect to a Python runtime: At the top-right of the menu bar, select *CONNECT*.
2. To run all the code in the notebook, select **Runtime** > **Run all**. To run the code cells one at a time, hover over each cell and select the **Run cell** icon.

![Run cell icon](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/images/beginner/run_cell_icon.png?raw=1)

## Another TensorFlow ML Training Example: Fashion MNIST

This example demonstrates training a simple convolutional neural network (CNN) to classify images from the Fashion MNIST dataset. Each image shows an article of clothing (e.g., T-shirt, sneaker, coat), and the goal is to correctly identify the type of clothing.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


### Load the Fashion MNIST dataset

The Fashion MNIST dataset consists of 70,000 grayscale images of clothing, split into 60,000 training images and 10,000 test images. Each image is 28x28 pixels, and there are 10 classes.

In [2]:
# Load the Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Define class names for plotting later
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training images shape: {train_images.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training images shape: (60000, 28, 28)
Training labels shape: (60000,)
Test images shape: (10000, 28, 28)
Test labels shape: (10000,)


### Preprocess the data

Before training, the image data needs to be preprocessed. The pixel values are currently integers from 0-255. We will scale them to a range of 0 to 1, which helps with model training. Also, for convolutional layers, the input often needs an explicit channel dimension (e.g., `(28, 28, 1)` for grayscale).

In [3]:
# Normalize pixel values to be between 0 and 1
train_images = train_images / 255.0
test_images = test_images / 255.0

# Reshape images to add a channel dimension, necessary for Conv2D layers
# (batch_size, height, width, channels)
train_images = train_images[..., tf.newaxis]
test_images = test_images[..., tf.newaxis]

print(f"Normalized and reshaped training images shape: {train_images.shape}")
print(f"Normalized and reshaped test images shape: {test_images.shape}")


Normalized and reshaped training images shape: (60000, 28, 28, 1)
Normalized and reshaped test images shape: (10000, 28, 28, 1)


### Build the Convolutional Neural Network (CNN) model

We will construct a simple CNN model. CNNs are particularly effective for image classification tasks. The model will consist of:
- `Conv2D` layers for feature extraction.
- `MaxPooling2D` layers for downsampling.
- `Flatten` to convert the 2D feature maps to a 1D vector.
- `Dense` layers for classification.

In [4]:
model_fashion = tf.keras.Sequential([
    # First convolutional block
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)), # 32 filters, 3x3 kernel
    tf.keras.layers.MaxPooling2D((2, 2)), # Reduce spatial dimensions

    # Second convolutional block
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'), # 64 filters, 3x3 kernel
    tf.keras.layers.MaxPooling2D((2, 2)), # Reduce spatial dimensions

    # Flatten the output of the convolutional layers to feed into dense layers
    tf.keras.layers.Flatten(),

    # Dense hidden layer with ReLU activation
    tf.keras.layers.Dense(128, activation='relu'),

    # Output layer with 10 units (for 10 classes) and no activation yet
    # We'll apply softmax in the loss function for numerical stability
    tf.keras.layers.Dense(10)
])

model_fashion.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

### Compile the model

Before training, the model needs to be compiled. This step configures the training process by specifying:
- An `optimizer`: The algorithm used to update model weights (e.g., 'adam').
- A `loss function`: Measures how accurate the model is during training (e.g., `SparseCategoricalCrossentropy` for integer labels).
- `Metrics`: Used to monitor the training and testing steps (e.g., 'accuracy').

In [5]:
# Define the loss function. `from_logits=True` because the output layer does not have a softmax activation.
loss_fn_fashion = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Compile the model
model_fashion.compile(optimizer='adam',
                      loss=loss_fn_fashion,
                      metrics=['accuracy'])


### Train the model

Training the model involves iterating over the training data multiple times (`epochs`). During each iteration, the model adjusts its internal parameters to minimize the loss function and improve accuracy.

In [6]:
epochs = 10 # Number of times to iterate over the entire training dataset

print(f"Training the model for {epochs} epochs...")
model_fashion.fit(train_images, train_labels, epochs=epochs)
print("Model training complete.")


Training the model for 10 epochs...
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 64s 33ms/step - accuracy: 0.7756 - loss: 0.6093
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 33ms/step - accuracy: 0.8859 - loss: 0.3137
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 62s 33ms/step - accuracy: 0.9038 - loss: 0.2620
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 33ms/step - accuracy: 0.9143 - loss: 0.2313
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9239 - loss: 0.2039
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 83s 32ms/step - accuracy: 0.9331 - loss: 0.1764
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9431 - loss: 0.1529
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 32ms/step - accuracy: 0.9495 - loss: 0.1365
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 32ms/step - accuracy: 0.9555 - loss: 0.1184
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9591 - loss: 0.1107
Model training complete.


### Evaluate the model

After training, it's important to evaluate the model's performance on unseen data (the test set) to ensure it generalizes well and hasn't just memorized the training data.

In [7]:
print("Evaluating model performance on the test set...")
test_loss, test_accuracy = model_fashion.evaluate(test_images, test_labels, verbose=2)

print(f'\nTest loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')


Evaluating model performance on the test set...
313/313 - 4s - 14ms/step - accuracy: 0.9064 - loss: 0.2922

Test loss: 0.2922
Test accuracy: 0.9064


### Make predictions

Once the model is trained, you can use it to make predictions on new images. The model's output layer provides logits, which can be converted to probabilities using `tf.nn.softmax`.

In [8]:
# Wrap the trained model with a Softmax layer to get probabilities
probability_model_fashion = tf.keras.Sequential([
    model_fashion,
    tf.keras.layers.Softmax()
])

# Make predictions on a few test images
predictions = probability_model_fashion.predict(test_images[:5])

# Display the predicted probabilities for the first test image
print("Predictions for the first 5 test images (probabilities for each class):\n")
for i, prediction in enumerate(predictions):
    print(f"Image {i+1}:")
    print(f"  Predicted class (index): {np.argmax(prediction)}")
    print(f"  Actual class (index): {test_labels[i]}")
    print(f"  Predicted class name: {class_names[np.argmax(prediction)]}")
    print(f"  Actual class name: {class_names[test_labels[i]]}")
    print(f"  Highest probability: {100*np.max(prediction):.2f}%\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Predictions for the first 5 test images (probabilities for each class):

Image 1:
  Predicted class (index): 9
  Actual class (index): 9
  Predicted class name: Ankle boot
  Actual class name: Ankle boot
  Highest probability: 100.00%

Image 2:
  Predicted class (index): 2
  Actual class (index): 2
  Predicted class name: Pullover
  Actual class name: Pullover
  Highest probability: 99.90%

Image 3:
  Predicted class (index): 1
  Actual class (index): 1
  Predicted class name: Trouser
  Actual class name: Trouser
  Highest probability: 100.00%

Image 4:
  Predicted class (index): 1
  Actual class (index): 1
  Predicted class name: Trouser
  Actual class name: Trouser
  Highest probability: 100.00%

Image 5:
  Predicted class (index): 6
  Actual class (index): 6
  Predicted class name: Shirt
  Actual class name: Shirt
  Highest probability: 74.00%



## Set up TensorFlow

Import TensorFlow into your program to get started:

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

If you are following along in your own development environment, rather than [Colab](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb), see the [install guide](https://www.tensorflow.org/install) for setting up TensorFlow for development.

Note: Make sure you have upgraded to the latest `pip` to install the TensorFlow 2 package if you are using your own development environment. See the [install guide](https://www.tensorflow.org/install) for details.

## Load a dataset

Load and prepare the MNIST dataset. The pixel values of the images range from 0 through 255. Scale these values to a range of 0 to 1 by dividing the values by `255.0`. This also converts the sample data from integers to floating-point numbers:

In [ ]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

## Build a machine learning model

Build a `tf.keras.Sequential` model:

In [9]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(64, activation='relu'), # Added another Dense layer
  tf.keras.layers.Dropout(0.3), # Increased dropout slightly for regularization
  tf.keras.layers.Dense(10)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) is useful for stacking layers where each layer has one input [tensor](https://www.tensorflow.org/guide/tensor) and one output tensor. Layers are functions with a known mathematical structure that can be reused and have trainable variables. Most TensorFlow models are composed of layers. This model uses the [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense), and [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) layers.

For each example, the model returns a vector of [logits](https://developers.google.com/machine-learning/glossary#logits) or [log-odds](https://developers.google.com/machine-learning/glossary#log-odds) scores, one for each class.

In [ ]:
predictions = model(x_train[:1]).numpy()
predictions

The `tf.nn.softmax` function converts these logits to *probabilities* for each class:

In [ ]:
tf.nn.softmax(predictions).numpy()

Note: It is possible to bake the `tf.nn.softmax` function into the activation function for the last layer of the network. While this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output.

Define a loss function for training using `losses.SparseCategoricalCrossentropy`:

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

The loss function takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. This loss is equal to the negative log probability of the true class: The loss is zero if the model is sure of the correct class.

This untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to `-tf.math.log(1/10) ~= 2.3`.

In [ ]:
loss_fn(y_train[:1], predictions).numpy()

Before you start training, configure and compile the model using Keras `Model.compile`. Set the [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) class to `adam`, set the `loss` to the `loss_fn` function you defined earlier, and specify a metric to be evaluated for the model by setting the `metrics` parameter to `accuracy`.

In [ ]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## Train and evaluate your model

Use the `Model.fit` method to adjust your model parameters and minimize the loss:

In [11]:
# Load and prepare the MNIST dataset (included here to ensure x_train, y_train are defined)
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Re-compile the model after defining or modifying its architecture
# This is necessary because the previous model definition was updated
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=10) # Increased epochs from 5 to 10

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


ValueError: You must call `compile()` before using the model.

The `Model.evaluate` method checks the model's performance, usually on a [validation set](https://developers.google.com/machine-learning/glossary#validation-set) or [test set](https://developers.google.com/machine-learning/glossary#test-set).

In [ ]:
model.evaluate(x_test,  y_test, verbose=2)

The image classifier is now trained to ~98% accuracy on this dataset. To learn more, read the [TensorFlow tutorials](https://www.tensorflow.org/tutorials/).

If you want your model to return a probability, you can wrap the trained model, and attach the softmax to it:

In [ ]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [ ]:
probability_model(x_test[:5])

## Conclusion

Congratulations! You have trained a machine learning model using a prebuilt dataset using the [Keras](https://www.tensorflow.org/guide/keras/overview) API.

For more examples of using Keras, check out the [tutorials](https://www.tensorflow.org/tutorials/keras/). To learn more about building models with Keras, read the [guides](https://www.tensorflow.org/guide/keras). If you want learn more about loading and preparing data, see the tutorials on [image data loading](https://www.tensorflow.org/tutorials/load_data/images) or [CSV data loading](https://www.tensorflow.org/tutorials/load_data/csv).
